# CL1 · Cloud para datos e IA

**Práctica guiada.** Vas a hacer, con **AWS simulado en memoria** (`moto` + `boto3`), lo que haces en una cuenta real: subir un modelo a S3 con versionado, escribir una política de mínimo privilegio, guardar secretos, probar el handler de una Lambda y calcular costes. Después tendrás las plantillas (CLI, Bicep, Terraform) para repetirlo en AWS y Azure de verdad.

> **Sin credenciales, sin red, sin coste.** La primera celda fija credenciales *falsas*: es imposible tocar una cuenta real por error. La nube real está en la guía del `README`.

Hilo conductor: el servicio de abandono de clientes de **TiendaSur** del módulo I8.

In [ ]:
import json
import os
import shutil
import time
from pathlib import Path

import boto3
import pandas as pd
import requests
from moto import mock_aws

# 1) Credenciales FALSAS y región fija: así es imposible tocar una cuenta real por error.
for var in ("AWS_PROFILE", "AWS_SESSION_TOKEN"):
    os.environ.pop(var, None)
os.environ.update(AWS_ACCESS_KEY_ID="clave-falsa", AWS_SECRET_ACCESS_KEY="secreto-falso",
                  AWS_DEFAULT_REGION="eu-west-1")
REGION = "eu-west-1"   # Irlanda. España tiene eu-south-2 (Aragón): comprueba la disponibilidad de cada servicio.

# 2) moto intercepta las llamadas de boto3 y las resuelve EN MEMORIA: sin red, sin cuenta, sin coste.
mock = mock_aws()
mock.start()

# Carpeta de trabajo del módulo (está en .gitignore del curso)
BASE = Path("proyecto-cloud").resolve()
if BASE.exists():
    shutil.rmtree(BASE)
BASE.mkdir()
print("moto activo; carpeta de trabajo:", BASE)
sts = boto3.client("sts")
print("Cuenta simulada:", sts.get_caller_identity()["Account"], "(no es una cuenta real)")

## 1 · El artefacto a desplegar

Un modelo en producción es **dos cosas**: el binario y su ficha (versión, métricas, umbral, limitaciones). Subiremos ambas.

In [ ]:
# Artefactos a subir: usamos el modelo y la ficha del módulo I8 si ya los generaste;
# si no, entrenamos uno diminuto para que este notebook funcione solo.
import joblib

I8 = Path("../I8-produccion/proyecto-churn/artefactos")
ARTEF = BASE / "artefactos"
ARTEF.mkdir()
if (I8 / "modelo.joblib").exists():
    shutil.copy(I8 / "modelo.joblib", ARTEF / "modelo.joblib")
    shutil.copy(I8 / "ficha.json", ARTEF / "ficha.json")
    print("Usando el modelo del módulo I8")
else:
    import numpy as np
    from sklearn.compose import ColumnTransformer
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder, StandardScaler

    rng = np.random.default_rng(0)
    n = 1500
    X = pd.DataFrame({
        "antiguedad_meses": rng.integers(1, 60, n), "pedidos_ultimo_ano": rng.integers(0, 40, n),
        "ticket_medio": rng.uniform(10, 120, n), "dias_desde_ultimo_pedido": rng.integers(0, 365, n),
        "incidencias_soporte": rng.integers(0, 6, n),
        "plan": rng.choice(["basico", "plus", "premium"], n), "canal": rng.choice(["web", "app", "tienda"], n)})
    y = (X.dias_desde_ultimo_pedido / 365 + X.incidencias_soporte / 6 + rng.normal(0, .3, n) > 1).astype(int)
    prep = ColumnTransformer([("num", StandardScaler(), [c for c in X.columns if c not in ("plan", "canal")]),
                              ("cat", OneHotEncoder(handle_unknown="ignore"), ["plan", "canal"])])
    joblib.dump(Pipeline([("prep", prep), ("clf", LogisticRegression(max_iter=500))]).fit(X, y), ARTEF / "modelo.joblib")
    (ARTEF / "ficha.json").write_text(json.dumps({"version_modelo": "demo-1", "umbral_decision": 0.5,
                                                  "columnas": list(X.columns)}, indent=2))
    print("Modelo de demostración entrenado")

ficha = json.loads((ARTEF / "ficha.json").read_text())
print("Versión del modelo:", ficha["version_modelo"], "· tamaño:", round((ARTEF / "modelo.joblib").stat().st_size / 1024), "KB")

## 2 · Almacenamiento de objetos (S3 / Blob Storage / Cloud Storage)

El almacenamiento de objetos es la base de casi todo: datos, modelos, logs, resultados. Tres ajustes de seguridad al crear un bucket: **bloquear acceso público**, **cifrado** y **versionado** (permite deshacer errores). Para compartir un archivo concreto temporalmente se usa una **URL prefirmada**, no un bucket público.

In [ ]:
s3 = boto3.client("s3")
BUCKET = "tiendasur-modelos-dev"

# Los buckets de regiones distintas de us-east-1 exigen indicar la región
s3.create_bucket(Bucket=BUCKET, CreateBucketConfiguration={"LocationConstraint": REGION})

# Tres ajustes de seguridad que hay que hacer SIEMPRE al crear un bucket:
s3.put_public_access_block(Bucket=BUCKET, PublicAccessBlockConfiguration={
    "BlockPublicAcls": True, "IgnorePublicAcls": True, "BlockPublicPolicy": True, "RestrictPublicBuckets": True})
s3.put_bucket_encryption(Bucket=BUCKET, ServerSideEncryptionConfiguration={"Rules": [
    {"ApplyServerSideEncryptionByDefault": {"SSEAlgorithm": "AES256"}}]})
s3.put_bucket_versioning(Bucket=BUCKET, VersioningConfiguration={"Status": "Enabled"})

# Subimos el modelo y la ficha (v1)
s3.upload_file(str(ARTEF / "modelo.joblib"), BUCKET, "churn/modelo.joblib")
s3.upload_file(str(ARTEF / "ficha.json"), BUCKET, "churn/ficha.json")

# Publicamos una "v2" de la ficha (por ejemplo, tras recalibrar el umbral)
ficha_v2 = {**ficha, "umbral_decision": 0.4, "nota": "umbral bajado a 0.4 tras revisar el coste de la campaña"}
s3.put_object(Bucket=BUCKET, Key="churn/ficha.json", Body=json.dumps(ficha_v2).encode())

versiones = s3.list_object_versions(Bucket=BUCKET, Prefix="churn/ficha.json")["Versions"]
for v in versiones:
    print(f"{v['Key']:20s} id={v['VersionId'][:8]}…  actual={v['IsLatest']}  {v['Size']} bytes")

In [ ]:
# Con el versionado activo, "borrar" o sobrescribir no destruye nada: podemos volver atrás.
anterior = next(v for v in versiones if not v["IsLatest"])
cuerpo = s3.get_object(Bucket=BUCKET, Key="churn/ficha.json", VersionId=anterior["VersionId"])["Body"].read()
print("Umbral en la versión anterior:", json.loads(cuerpo)["umbral_decision"])
print("Umbral en la versión actual:  ", json.loads(s3.get_object(Bucket=BUCKET, Key="churn/ficha.json")["Body"].read())["umbral_decision"])

# URL prefirmada: acceso temporal a UN objeto sin dar credenciales ni hacer público el bucket.
url = s3.generate_presigned_url("get_object", Params={"Bucket": BUCKET, "Key": "churn/ficha.json"}, ExpiresIn=300)
print("\nURL prefirmada (caduca en 300 s):", url.split("?")[0], "+ firma…")
r = requests.get(url)
print("Descarga con la URL:", r.status_code, "· versión del modelo:", r.json()["version_modelo"])

## 3 · IAM y mínimo privilegio

Regla de oro: cada servicio o persona recibe **solo** los permisos que necesita, sobre **solo** los recursos que necesita. IAM evalúa así: todo denegado por defecto → un `Allow` que coincida lo permite → un `Deny` explícito siempre gana. Escribimos una función propia que imita esa lógica (simplificada) para *entender* qué permite una política antes de aplicarla.

En Azure es igual en concepto pero distinto en forma: asignas **roles** (RBAC) a una identidad (`Entra ID`, o una *identidad gestionada* para servicios) sobre un **ámbito** (suscripción, grupo de recursos o recurso).

In [ ]:
import fnmatch

# Política de MÍNIMO PRIVILEGIO para la función que sirve el modelo:
# solo puede LEER objetos bajo churn/ y leer UN secreto. Nada más.
ARN_BUCKET = f"arn:aws:s3:::{BUCKET}"
POLITICA_LECTOR = {
    "Version": "2012-10-17",
    "Statement": [
        {"Sid": "LeerModelo", "Effect": "Allow", "Action": ["s3:GetObject"],
         "Resource": [f"{ARN_BUCKET}/churn/*"]},
        {"Sid": "LeerSecreto", "Effect": "Allow", "Action": ["secretsmanager:GetSecretValue"],
         "Resource": ["arn:aws:secretsmanager:eu-west-1:*:secret:tiendasur/churn/*"]},
        {"Sid": "NuncaBorrar", "Effect": "Deny", "Action": ["s3:Delete*"], "Resource": ["*"]},
    ],
}

def _lista(x):
    return x if isinstance(x, list) else [x]

def evaluar(politica: dict, accion: str, recurso: str) -> bool:
    """Versión simplificada de cómo decide IAM: (1) todo está denegado por defecto,
    (2) un Allow que coincida lo permite, (3) un Deny explícito que coincida SIEMPRE gana."""
    permitido = False
    for st in politica["Statement"]:
        coincide = (any(fnmatch.fnmatchcase(accion, a) for a in _lista(st["Action"]))
                    and any(fnmatch.fnmatchcase(recurso, r) for r in _lista(st["Resource"])))
        if coincide and st["Effect"] == "Deny":
            return False
        if coincide and st["Effect"] == "Allow":
            permitido = True
    return permitido

pruebas = [
    ("s3:GetObject",    f"{ARN_BUCKET}/churn/modelo.joblib", True),
    ("s3:GetObject",    f"{ARN_BUCKET}/otros/datos-clientes.csv", False),   # fuera del prefijo
    ("s3:PutObject",    f"{ARN_BUCKET}/churn/modelo.joblib", False),        # no puede escribir
    ("s3:DeleteObject", f"{ARN_BUCKET}/churn/modelo.joblib", False),        # Deny explícito
    ("secretsmanager:GetSecretValue", "arn:aws:secretsmanager:eu-west-1:123456789012:secret:tiendasur/churn/api-AbCdEf", True),
    ("iam:CreateUser",  "*", False),
]
tabla = pd.DataFrame([(a, r.replace(ARN_BUCKET, "bucket"), esperado, evaluar(POLITICA_LECTOR, a, r))
                      for a, r, esperado in pruebas], columns=["acción", "recurso", "esperado", "resultado"])
tabla["ok"] = tabla.esperado == tabla.resultado
print(tabla.to_string(index=False))
assert tabla.ok.all(), "alguna regla no se comporta como se esperaba"

# Crear de verdad (en la nube simulada) la política y un rol que la Lambda puede asumir
iam = boto3.client("iam")
confianza = {"Version": "2012-10-17", "Statement": [{"Effect": "Allow",
             "Principal": {"Service": "lambda.amazonaws.com"}, "Action": "sts:AssumeRole"}]}
rol = iam.create_role(RoleName="churn-lambda", AssumeRolePolicyDocument=json.dumps(confianza))["Role"]
pol = iam.create_policy(PolicyName="churn-lector", PolicyDocument=json.dumps(POLITICA_LECTOR))["Policy"]
iam.attach_role_policy(RoleName="churn-lambda", PolicyArn=pol["Arn"])
print("\nRol:", rol["Arn"])
print("Políticas adjuntas:", [p["PolicyName"] for p in iam.list_attached_role_policies(RoleName="churn-lambda")["AttachedPolicies"]])

## 4 · Secretos

Las claves van en un gestor de secretos (Secrets Manager, Key Vault, Secret Manager), no en el código ni en variables del `Dockerfile`. La aplicación obtiene permiso por su **rol** (el de la sección anterior) y lee el secreto al arrancar.

In [ ]:
sm = boto3.client("secretsmanager")
# Un secreto es un JSON: así guardas varias claves relacionadas bajo un único nombre.
sm.create_secret(Name="tiendasur/churn/api",
                 SecretString=json.dumps({"api_key": "sk-ejemplo-no-es-real", "db_password": "cambia-esto"}),
                 Description="Credenciales de la API de churn (demo)")

# La aplicación lee el secreto EN TIEMPO DE EJECUCIÓN, nunca va en el código, ni en la imagen, ni en Git.
valor = json.loads(sm.get_secret_value(SecretId="tiendasur/churn/api")["SecretString"])
print("Claves disponibles:", list(valor))          # imprime los NOMBRES, nunca los valores
print("Longitud de api_key:", len(valor["api_key"]))

# Rotar = guardar una nueva versión; la antigua queda con la etiqueta AWSPREVIOUS
sm.put_secret_value(SecretId="tiendasur/churn/api",
                    SecretString=json.dumps({**valor, "api_key": "sk-ejemplo-rotada"}))
versiones_sec = sm.list_secret_version_ids(SecretId="tiendasur/churn/api")["Versions"]
print("Versiones y etiquetas:", [v.get("VersionStages") for v in versiones_sec])

## 5 · Cómputo sin servidor (Lambda / Functions / Cloud Run functions)

Una función se ejecuta solo cuando llega un evento y se cobra por uso. Dos ideas clave que verás en la salida: el **arranque en frío** (la primera invocación carga el modelo; las siguientes lo reutilizan) y que el *handler* es una función normal de Python, **testeable en local** sin desplegar nada.

> Nota práctica: un modelo con scikit-learn supera con facilidad el límite de un paquete `.zip` de Lambda. Para modelos reales se despliega la función como **imagen de contenedor** (hasta 10 GB), la misma idea que el Dockerfile del módulo I8.

In [ ]:
# El código de la función vive en un archivo (así lo empaquetarías). Lo escribimos y lo importamos.
(BASE / "lambda").mkdir()
(BASE / "lambda" / "handler.py").write_text(r"""
import json
import os
import tempfile

import boto3
import joblib
import pandas as pd

BUCKET = os.environ.get("MODEL_BUCKET", "tiendasur-modelos-dev")
CLAVE = os.environ.get("MODEL_KEY", "churn/modelo.joblib")
CAMPOS = ["antiguedad_meses", "pedidos_ultimo_ano", "ticket_medio", "dias_desde_ultimo_pedido",
          "incidencias_soporte", "plan", "canal"]

s3 = boto3.client("s3")          # FUERA del handler: se crea una vez por entorno de ejecución
_modelo = None                   # caché entre invocaciones "calientes"


def _cargar_modelo():
    global _modelo
    if _modelo is None:          # solo en el "arranque en frío" (cold start)
        with tempfile.NamedTemporaryFile(suffix=".joblib") as f:
            s3.download_file(BUCKET, CLAVE, f.name)
            _modelo = joblib.load(f.name)
    return _modelo


def _respuesta(codigo, cuerpo):
    return {"statusCode": codigo, "headers": {"Content-Type": "application/json"}, "body": json.dumps(cuerpo)}


def handler(event, context=None):
    try:
        datos = json.loads(event.get("body") or "{}")
    except json.JSONDecodeError:
        return _respuesta(400, {"error": "el cuerpo no es JSON válido"})
    faltan = [c for c in CAMPOS if c not in datos]
    if faltan:
        return _respuesta(422, {"error": "faltan campos", "campos": faltan})
    p = float(_cargar_modelo().predict_proba(pd.DataFrame([datos]))[0, 1])
    return _respuesta(200, {"probabilidad_abandono": round(p, 4)})
""")

import importlib.util, sys
espec = importlib.util.spec_from_file_location("handler_churn", BASE / "lambda" / "handler.py")
handler_churn = importlib.util.module_from_spec(espec); espec.loader.exec_module(handler_churn)

cliente = {"antiguedad_meses": 8, "pedidos_ultimo_ano": 3, "ticket_medio": 35.0,
           "dias_desde_ultimo_pedido": 200, "incidencias_soporte": 3, "plan": "basico", "canal": "web"}
evento = {"body": json.dumps(cliente)}

t0 = time.perf_counter(); r1 = handler_churn.handler(evento); frio = time.perf_counter() - t0
t0 = time.perf_counter(); r2 = handler_churn.handler(evento); caliente = time.perf_counter() - t0
print("1ª invocación (arranque en frío):", r1["statusCode"], r1["body"], f"· {frio*1000:.0f} ms")
print("2ª invocación (caliente):        ", r2["statusCode"], r2["body"], f"· {caliente*1000:.0f} ms")
print("Evento incompleto ->", handler_churn.handler({"body": json.dumps({"plan": "plus"})})["statusCode"])
print("Cuerpo no JSON    ->", handler_churn.handler({"body": "hola"})["statusCode"])
assert r1["statusCode"] == 200 and caliente < frio

## 6 · Costes: pago por uso frente a instancia fija

Los precios de abajo son **orientativos** (comprueba siempre las calculadoras oficiales). Lo importante no es el número sino la **forma**: pago por uso es casi gratis con poco tráfico y crece linealmente; la instancia fija cuesta lo mismo con 0 o con millones de peticiones, hasta que se satura.

In [ ]:
# PRECIOS ORIENTATIVOS (dólares, región de EE. UU./Europa, orden de magnitud de 2025-2026).
# NO los uses para presupuestar: comprueba la calculadora oficial de cada proveedor.
PRECIOS = {
    "lambda_por_millon_peticiones": 0.20,
    "lambda_por_gb_segundo": 0.0000166667,
    "vm_pequena_por_hora": 0.021,        # instancia pequeña siempre encendida (2 GB)
    "horas_mes": 730,
}
CAPA_GRATIS_LAMBDA = {"peticiones": 1_000_000, "gb_segundo": 400_000}   # mensual; comprueba vigencia

def coste_lambda(peticiones_mes, ms_por_peticion=120, memoria_mb=512, con_capa_gratis=False):
    gb_s = peticiones_mes * (ms_por_peticion / 1000) * (memoria_mb / 1024)
    peticiones_cobradas = peticiones_mes
    if con_capa_gratis:
        gb_s = max(0, gb_s - CAPA_GRATIS_LAMBDA["gb_segundo"])
        peticiones_cobradas = max(0, peticiones_mes - CAPA_GRATIS_LAMBDA["peticiones"])
    return peticiones_cobradas / 1e6 * PRECIOS["lambda_por_millon_peticiones"] + gb_s * PRECIOS["lambda_por_gb_segundo"]

def coste_instancia_fija(n_instancias=1):
    return n_instancias * PRECIOS["vm_pequena_por_hora"] * PRECIOS["horas_mes"]

def punto_de_equilibrio(**kw):
    """Peticiones/mes a partir de las cuales la instancia fija sale más barata que pagar por uso."""
    fijo = coste_instancia_fija()
    lo, hi = 1, 10**10
    while lo < hi:
        medio = (lo + hi) // 2
        if coste_lambda(medio, **kw) < fijo:
            lo = medio + 1
        else:
            hi = medio
    return lo

volumenes = [10_000, 100_000, 1_000_000, 10_000_000, 100_000_000]
tabla = pd.DataFrame({"peticiones/mes": volumenes,
                      "Lambda ($)": [round(coste_lambda(v), 2) for v in volumenes],
                      "Lambda con capa gratis ($)": [round(coste_lambda(v, con_capa_gratis=True), 2) for v in volumenes],
                      "1 instancia fija ($)": round(coste_instancia_fija(), 2)})
print(tabla.to_string(index=False))
print(f"\nPunto de equilibrio: ~{punto_de_equilibrio():,} peticiones/mes (120 ms, 512 MB)")
print("Con un modelo que tarda 800 ms:", f"~{punto_de_equilibrio(ms_por_peticion=800):,}")

In [ ]:
# Alarma de presupuesto: lo PRIMERO que se configura en una cuenta nueva.
bud = boto3.client("budgets")
cuenta = sts.get_caller_identity()["Account"]
bud.create_budget(AccountId=cuenta, Budget={
    "BudgetName": "tope-curso-5-usd", "BudgetType": "COST", "TimeUnit": "MONTHLY",
    "BudgetLimit": {"Amount": "5", "Unit": "USD"}},
    NotificationsWithSubscribers=[
        {"Notification": {"NotificationType": "ACTUAL", "ComparisonOperator": "GREATER_THAN",
                          "Threshold": 80, "ThresholdType": "PERCENTAGE"},
         "Subscribers": [{"SubscriptionType": "EMAIL", "Address": "tu-correo@ejemplo.com"}]},
        {"Notification": {"NotificationType": "FORECASTED", "ComparisonOperator": "GREATER_THAN",
                          "Threshold": 100, "ThresholdType": "PERCENTAGE"},
         "Subscribers": [{"SubscriptionType": "EMAIL", "Address": "tu-correo@ejemplo.com"}]}])
print([b["BudgetName"] for b in bud.describe_budgets(AccountId=cuenta)["Budgets"]])
print("Avisa al 80 % de lo gastado y cuando la PREVISIÓN supere el 100 %. Un presupuesto AVISA, no corta el gasto.")

## 7 · Del patrón común a cada nube

Aprender un concepto una vez y traducirlo: esta tabla es tu chuleta en una entrevista.

In [ ]:
equivalencias = pd.DataFrame([
    ("Almacenamiento de objetos", "S3",                    "Blob Storage",                 "Cloud Storage"),
    ("Identidad y permisos",       "IAM (usuarios, roles)", "Entra ID + RBAC",              "Cloud IAM"),
    ("Secretos",                   "Secrets Manager",       "Key Vault",                    "Secret Manager"),
    ("Funciones sin servidor",     "Lambda",                "Functions",                    "Cloud Run functions"),
    ("Registro de imágenes",       "ECR",                   "Container Registry (ACR)",     "Artifact Registry"),
    ("Contenedor gestionado (simple)", "App Runner",        "Container Apps",               "Cloud Run"),
    ("Contenedores orquestados",   "ECS Fargate / EKS",     "Container Apps / AKS",         "Cloud Run / GKE"),
    ("Observabilidad",             "CloudWatch",            "Monitor + Application Insights", "Cloud Monitoring / Logging"),
    ("Presupuestos y costes",      "Budgets + Cost Explorer", "Cost Management + Budgets", "Billing budgets"),
    ("LLMs gestionados",           "Bedrock",               "Foundry (Azure OpenAI y otros)", "Vertex AI (Model Garden)"),
    ("Plataforma de ML",           "SageMaker",             "Azure Machine Learning",       "Vertex AI"),
    ("Infraestructura como código nativa", "CloudFormation / CDK", "Bicep / ARM",          "Deployment Manager / Terraform"),
], columns=["Necesidad", "AWS", "Azure", "GCP"])
print(equivalencias.to_string(index=False))

## 8 · Infraestructura como código y plantillas de despliegue

Se escriben en `proyecto-cloud/` **sin aplicarse**. Léelas: son exactamente los pasos de la guía del `README` para desplegar la imagen del módulo I8 en **AWS App Runner** y en **Azure Container Apps**, con el comando para **apagarlo todo** al final. `Terraform` describe lo mismo de forma declarativa y multinube (`plan` te muestra qué cambiaría antes de tocar nada; `destroy` lo borra).

In [ ]:
# Escribimos las plantillas en proyecto-cloud/ para que las abras y las leas. NO se ejecutan.
def escribir_archivo(ruta, contenido):
    destino = BASE / ruta
    destino.parent.mkdir(parents=True, exist_ok=True)
    destino.write_text(contenido.lstrip("\n"))
    print("escrito:", destino.relative_to(BASE.parent))

escribir_archivo('aws/desplegar-apprunner.sh', r"""
#!/usr/bin/env bash
# PLANTILLA DE LECTURA: despliegue del contenedor de I8 en AWS App Runner. No se ejecuta desde el notebook.
# Requisitos: AWS CLI v2 configurada con un usuario/rol SIN permisos de administrador (IAM Identity Center),
# Docker, y haber creado la alarma de presupuesto (ver guia). Sustituye los valores entre <>.
set -euo pipefail
REGION=eu-west-1
CUENTA=$(aws sts get-caller-identity --query Account --output text)
REPO=tiendasur-churn
IMAGEN=$CUENTA.dkr.ecr.$REGION.amazonaws.com/$REPO:v1

# 1) Registro de imagenes
aws ecr create-repository --repository-name $REPO --region $REGION --image-scanning-configuration scanOnPush=true
aws ecr get-login-password --region $REGION | docker login --username AWS --password-stdin $CUENTA.dkr.ecr.$REGION.amazonaws.com

# 2) Construir (para la arquitectura del servicio: linux/amd64) y subir la imagen del modulo I8
docker build --platform linux/amd64 -t $IMAGEN ../../I8-produccion/proyecto-churn
docker push $IMAGEN

# 3) Rol que permite a App Runner descargar de ECR (politica gestionada AWSAppRunnerServicePolicyForECRAccess)
aws iam create-role --role-name apprunner-ecr --assume-role-policy-document file://confianza-apprunner.json
aws iam attach-role-policy --role-name apprunner-ecr \
  --policy-arn arn:aws:iam::aws:policy/service-role/AWSAppRunnerServicePolicyForECRAccess

# 4) Servicio (0,25 vCPU / 0,5 GB: el minimo) con comprobacion de salud en /health
aws apprunner create-service --service-name churn-api --region $REGION \
  --source-configuration "{\"ImageRepository\":{\"ImageIdentifier\":\"$IMAGEN\",\"ImageRepositoryType\":\"ECR\",\"ImageConfiguration\":{\"Port\":\"8000\"}},\"AuthenticationConfiguration\":{\"AccessRoleArn\":\"arn:aws:iam::$CUENTA:role/apprunner-ecr\"}}" \
  --instance-configuration Cpu=256,Memory=512 \
  --health-check-configuration Protocol=HTTP,Path=/health

# 5) Probar
aws apprunner list-services --region $REGION --query "ServiceSummaryList[?ServiceName=='churn-api'].ServiceUrl" --output text
# curl https://<url>/health

# ---------------- APAGAR TODO (para no pagar) ----------------
# aws apprunner delete-service --service-arn <arn> --region $REGION
# aws ecr delete-repository --repository-name $REPO --force --region $REGION
# aws iam detach-role-policy --role-name apprunner-ecr --policy-arn arn:aws:iam::aws:policy/service-role/AWSAppRunnerServicePolicyForECRAccess
# aws iam delete-role --role-name apprunner-ecr

""")
escribir_archivo('aws/confianza-apprunner.json', r"""
{
  "Version": "2012-10-17",
  "Statement": [
    { "Effect": "Allow", "Principal": { "Service": "build.apprunner.amazonaws.com" }, "Action": "sts:AssumeRole" }
  ]
}
""")
escribir_archivo('aws/main.tf', r"""
# Terraform minimo: servicio App Runner para la imagen del modulo I8. EJEMPLO DE LECTURA, no se aplica.
# Flujo real:  terraform init -> terraform plan (lee el plan!) -> terraform apply -> terraform destroy
terraform {
  required_providers {
    aws = { source = "hashicorp/aws", version = "~> 5.0" }   # comprueba la version vigente
  }
}

provider "aws" { region = "eu-west-1" }

variable "imagen" {
  description = "URI de la imagen en ECR, p. ej. 123456789012.dkr.ecr.eu-west-1.amazonaws.com/tiendasur-churn:v1"
  type        = string
}

resource "aws_ecr_repository" "churn" {
  name = "tiendasur-churn"
  image_scanning_configuration { scan_on_push = true }
}

data "aws_iam_policy_document" "confianza" {
  statement {
    actions = ["sts:AssumeRole"]
    principals {
      type        = "Service"
      identifiers = ["build.apprunner.amazonaws.com"]
    }
  }
}

resource "aws_iam_role" "acceso_ecr" {
  name               = "apprunner-ecr"
  assume_role_policy = data.aws_iam_policy_document.confianza.json
}

resource "aws_iam_role_policy_attachment" "acceso_ecr" {
  role       = aws_iam_role.acceso_ecr.name
  policy_arn = "arn:aws:iam::aws:policy/service-role/AWSAppRunnerServicePolicyForECRAccess"
}

resource "aws_apprunner_service" "churn" {
  service_name = "churn-api"
  source_configuration {
    authentication_configuration { access_role_arn = aws_iam_role.acceso_ecr.arn }
    image_repository {
      image_identifier      = var.imagen
      image_repository_type = "ECR"
      image_configuration { port = "8000" }
    }
  }
  instance_configuration {
    cpu    = "256"
    memory = "512"
  }
  health_check_configuration {
    protocol = "HTTP"
    path     = "/health"
  }
}

output "url" { value = aws_apprunner_service.churn.service_url }

""")
escribir_archivo('azure/desplegar-containerapps.sh', r"""
#!/usr/bin/env bash
# PLANTILLA DE LECTURA: el mismo despliegue en Azure Container Apps. No se ejecuta desde el notebook.
# Requisitos: Azure CLI (az) con sesion iniciada (az login) y una suscripcion con credito gratuito.
set -euo pipefail
RG=rg-tiendasur-churn
LOC=westeurope
ACR=tiendasurchurn$RANDOM          # el nombre del registro debe ser unico globalmente y solo alfanumerico
ENTORNO=env-churn
APP=churn-api

# 1) Grupo de recursos: la "carpeta" que contiene todo y que se borra de una vez
az group create --name $RG --location $LOC

# 2) Registro de imagenes (ACR) y construccion EN la nube (sin Docker local)
az acr create --resource-group $RG --name $ACR --sku Basic
az acr build --registry $ACR --image churn:v1 ../../I8-produccion/proyecto-churn

# 3) Entorno de Container Apps y la aplicacion (escala a 0 cuando no hay trafico)
az containerapp env create --name $ENTORNO --resource-group $RG --location $LOC
az containerapp create --name $APP --resource-group $RG --environment $ENTORNO \
  --image $ACR.azurecr.io/churn:v1 --registry-server $ACR.azurecr.io --registry-identity system \
  --target-port 8000 --ingress external --min-replicas 0 --max-replicas 2 --cpu 0.25 --memory 0.5Gi

# 4) Probar
az containerapp show --name $APP --resource-group $RG --query properties.configuration.ingress.fqdn -o tsv
# curl https://<fqdn>/health

# 5) Alarma de presupuesto (Cost Management): mejor desde el portal la primera vez, o con
#    az consumption budget create ...  (comprueba la sintaxis vigente con az consumption budget --help)

# ---------------- APAGAR TODO (para no pagar) ----------------
# Borrar el grupo de recursos elimina TODO lo que contiene, incluido el registro y el entorno:
# az group delete --name $RG --yes --no-wait

""")
escribir_archivo('azure/main.bicep', r"""
// Bicep (lenguaje declarativo de Azure): el equivalente a CloudFormation. Plantilla de lectura, no se aplica aqui.
// Uso:  az deployment group create -g rg-tiendasur-churn -f main.bicep -p imagen=<acr>.azurecr.io/churn:v1
param location string = resourceGroup().location
param imagen string
param nombre string = 'churn-api'

resource workspace 'Microsoft.OperationalInsights/workspaces@2023-09-01' = {
  name: 'log-${nombre}'
  location: location
  properties: { sku: { name: 'PerGB2018' }, retentionInDays: 30 }
}

resource entorno 'Microsoft.App/managedEnvironments@2024-03-01' = {
  name: 'env-${nombre}'
  location: location
  properties: {
    appLogsConfiguration: {
      destination: 'log-analytics'
      logAnalyticsConfiguration: {
        customerId: workspace.properties.customerId
        sharedKey: workspace.listKeys().primarySharedKey
      }
    }
  }
}

resource app 'Microsoft.App/containerApps@2024-03-01' = {
  name: nombre
  location: location
  identity: { type: 'SystemAssigned' }          // identidad gestionada: sin contrasenas en el codigo
  properties: {
    managedEnvironmentId: entorno.id
    configuration: { ingress: { external: true, targetPort: 8000 } }
    template: {
      containers: [ { name: nombre, image: imagen, resources: { cpu: json('0.25'), memory: '0.5Gi' } } ]
      scale: { minReplicas: 0, maxReplicas: 2 }
    }
  }
}

output url string = 'https://${app.properties.configuration.ingress.fqdn}'

""")
escribir_archivo('azure/main.tf', r"""
# Terraform minimo: Azure Container Apps para la misma imagen. EJEMPLO DE LECTURA, no se aplica.
terraform {
  required_providers {
    azurerm = { source = "hashicorp/azurerm", version = "~> 4.0" }   # comprueba la version vigente
  }
}

provider "azurerm" {
  features {}
  subscription_id = var.suscripcion
}

variable "suscripcion" { type = string }
variable "imagen"      { type = string }

resource "azurerm_resource_group" "rg" {
  name     = "rg-tiendasur-churn"
  location = "westeurope"
}

resource "azurerm_container_app_environment" "env" {
  name                = "env-churn"
  location            = azurerm_resource_group.rg.location
  resource_group_name = azurerm_resource_group.rg.name
}

resource "azurerm_container_app" "churn" {
  name                         = "churn-api"
  container_app_environment_id = azurerm_container_app_environment.env.id
  resource_group_name          = azurerm_resource_group.rg.name
  revision_mode                = "Single"

  template {
    min_replicas = 0
    max_replicas = 2
    container {
      name   = "churn"
      image  = var.imagen
      cpu    = 0.25
      memory = "0.5Gi"
    }
  }

  ingress {
    external_enabled = true
    target_port      = 8000
    traffic_weight {
      percentage      = 100
      latest_revision = true
    }
  }
}

output "url" { value = "https://${azurerm_container_app.churn.ingress[0].fqdn}" }

""")
equivalencias.to_csv(BASE / 'equivalencias-nube.csv', index=False)

# Comprobación de higiene: ninguna plantilla debe contener credenciales
sospechosas = ("AKIA", "SECRET", "password=", "secret_key")
for p in sorted(BASE.rglob("*")):
    if p.is_file() and p.suffix in (".sh", ".tf", ".bicep", ".json"):
        texto = p.read_text()
        assert not any(s in texto for s in sospechosas), p
print("\nPlantillas revisadas: sin credenciales incrustadas")

## 9 · Cierre

## Ejercicios

Resuelve cada ejercicio en su celda (las celdas anteriores deben estar ejecutadas; moto sigue activo hasta la penúltima celda).

1. **Mínimo privilegio.** Escribe `POLITICA_ANALISTA`: puede *listar* el bucket (`s3:ListBucket` sobre el ARN del bucket) y *leer* (`s3:GetObject`) solo `churn/ficha.json`, nunca el modelo. Comprueba con `evaluar` tres casos permitidos/denegados.
2. **Retroceder un modelo.** Sube una «v2» de `churn/modelo.joblib` (cualquier contenido), lista sus versiones y *restaura* la anterior copiándola como versión actual (`copy_object` con `VersionId`). Comprueba que el tamaño vuelve al original.
3. **Handler robusto.** Prueba el handler con `plan: "oro"` (categoría desconocida). ¿Qué devuelve? Escribe una función `validar_evento(datos)` que devuelva una lista de errores (plan/canal permitidos, números no negativos) y úsala para devolver `422` antes de llamar al modelo.
4. **Costes.** TiendaSur espera 3 millones de predicciones al mes, 300 ms cada una y 1 GB de memoria. Calcula el coste en Lambda y en una instancia fija; decide y justifica en una frase (incluye el riesgo que no aparece en la fórmula).
5. **Traducir entre nubes.** Completa `traducir(servicio_aws, nube)` con la tabla `equivalencias` y comprueba que `traducir("Lambda", "Azure")` devuelve `Functions` y `traducir("ECR", "GCP")` devuelve `Artifact Registry`.

In [ ]:
# Ejercicio 1 · tu código aquí


In [ ]:
# Ejercicio 2 · tu código aquí


In [ ]:
# Ejercicio 3 · tu código aquí


In [ ]:
# Ejercicio 4 · tu código aquí


In [ ]:
# Ejercicio 5 · tu código aquí


In [ ]:
# Al final de la sesión: paramos la nube simulada (en una cuenta real, aquí iría "apagar y borrar").
mock.stop()
print("moto detenido: todo lo creado (buckets, roles, secretos) ha desaparecido de la memoria.")
print("En la nube real NADA desaparece solo: por eso existe la lista de «apagar» de cada guía.")